<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/W2_EDA_with_Airbnb_Listing_Data_shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCH-MGMT 661: Applications of AI Models  
**Instructor:** Indika Dissanayake  

---

## Tutorial 1: Data Preparation and Exploratory Data Analysis (EDA)  
**Dataset:** Inside Airbnb – Asheville Listings

---

This tutorial will guide you through the essential steps of data preparation and exploratory data analysis using the **Pandas** and **Seaborn** libraries in Python. You will learn how to:

- Load and explore a real-world dataset  
- Identify and handle missing data  
- Understand the structure and types of variables  
- Prepare the dataset for future modeling

We will be working in **Google Colab**, using an open dataset from **Inside Airbnb**, which provides transparent data on Airbnb listings across cities.


##1. Data Loading and Initial Exploration

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")



In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/listings.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')



### Exercise 1: Explore the Airbnb Dataset

Use the following commands to get an initial overview of the data:

```python
listings_df.head()     # Display the first few rows  
listings_df.info()     # Explore columns, data types, and missing values  
listings_df.shape      # Shape of the DataFrame (rows, columns)  


#### 🔍 What to Look For:

- **Dataset structure**: What kinds of columns are present? Do they look like numeric, categorical, or text?
- **Important variables**: Can you identify columns that might be useful for analysis, such as `price`, `room_type`, or `number_of_reviews`?
- **Missing data**: Are any columns showing a high number of missing (`null`) values?
- **Data types**: Are the data types appropriate for each column (e.g., numeric for prices)?
- **Size of the dataset**: How many rows and columns does the dataset contain?


In [ ]:
# Ex1 a: Display the first 10 rows



In [ ]:
# Ex1 b: Explore columns, data types, and non-null counts



In [ ]:
# Ex1 c: Display the shape of the DataFrame (rows, columns)


##2. Data Selection






### Filtering by Host Location

This dataset comes from Inside Airbnb and includes only listings located in Asheville, NC.
However, the dataset also contains information about where hosts are based. The host_location column reflects the host’s reported home location, which may differ from the location of the Airbnb unit.

---

### Exercise 2: Explore Unique Host Locations

```python
# Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations
```

### Exercise 3: Focus on Listings Managed by Local Hosts

In some analyses, we may want to focus specifically on listings managed by **local Asheville-based hosts**, rather than remote or out-of-state hosts.

There are multiple ways to filter the dataset based on the `host_location` column:

```python
# Option 1: Exact match (case-sensitive)
asheville_df = listings_df[listings_df['host_location'] == 'Asheville, NC']

# Option 2: Case-insensitive match using string contains
asheville_df = listings_df[listings_df['host_location'].str.contains('Asheville', case=False, na=False)]


#### 🔍 What to Look For:

- **Local vs. remote hosts**: Are Asheville listings mostly managed by local hosts or by hosts living elsewhere?
- **Location formatting**: Are the city names consistent (e.g., "Asheville, NC" vs. "Asheville")?
- **Missing data**: Are there any `NaN` values in the `host_location` column?
- **Need for data cleaning**: Is the `host_location` column reliable for filtering, or might it require additional cleaning?
- **Subset success**: After filtering, does the new DataFrame (`asheville_df`) contain only Asheville listings by local hosts? What is the shapeof the filtered dataset


In [ ]:
# Ex 2: Print all unique host locations in the dataset



In [ ]:
# Ex 3: Filter Listings for Asheville Hosts



In [ ]:
# Shape of the filtered dataset


### Subset by Columns: Selecting Relevant Features

Now that we’ve filtered the dataset for Asheville listings by local hosts, let’s simplify it further by selecting a smaller set of **useful columns** for analysis.

We’ll keep:
- **Numerical columns**: e.g., `price`, `bathrooms`,`bedrooms`, `number_of_reviews`, `latitude`, `longitude`
- **Categorical columns**: e.g., `room_type`, `host_identity_verified`, `host_is_superhost`

---

### 🧪 Exercise 4: Create a Subset of Columns

```python
# Define the columns we want to keep
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews', 'latitude',
    'longitude', 'room_type', 'host_identity_verified', 'host_is_superhost'
]

# Create a new DataFrame with only the selected columns
asheville_df = asheville_df[selected_columns]

# Preview the first few rows of the subset
asheville_df.head()


#### 🔍 What to Look For:

- **Missing values**: Do any columns like `bathrooms`, `bedrooms`, or `number_of_reviews` have missing entries?
- **Inconsistent formats**: Is the `price` column stored as a string or object (e.g., with `$`) instead of a numeric type?
- **Data types**: Are numeric-looking columns like `bathrooms`, `bedrooms`, and `price` actually of the correct data type?
- **Binary columns**: Are values in `host_is_superhost` and `host_identity_verified` stored as `'t'`/`'f'`? Would it be better to convert them to `1`/`0`?
- **Categorical columns**: Is `room_type` ready for modeling, or does it need to be one-hot encoded?
- **Geolocation columns**: Are `latitude` and `longitude` necessary for your analysis, or could they be dropped or used later for mapping?

These observations will guide how you clean, transform, and prepare your data in the next step: **Data Pre-processing**.



In [ ]:
#Ex 4:  Define the columns we want to keep and create a new dataframe with the selected columns


In [ ]:
# Preview the first few rows of the new dataframe


##3. Data Pre Processing
**(Handling missing values, Correcting Data Types, Data Transformation)**

### Exercise 5: Handling Missing Values

Missing data is common in real-world datasets. Before modeling or analysis, we need to decide how to handle it in a way that makes sense for the variable type and purpose.

---

### 5.1: Check for Missing Values

```python
# Count missing values in each column
asheville_df.isnull().sum()
```
---
### 5.2: Dropping Columns or Rows with Missing Values

Since we'll use price as the target variable for a prediction model in the next unit, let's not impute missing price values. Imputing the target could distort model training. Let's drop the records with missing price values. Then we can check the dataframe again for the missing values

```python
# Drop rows where the target variable (price) is missing
asheville_df = asheville_df.dropna(subset=['price'])

# Count missing values in each column
asheville_df.isnull().sum()
```

---


### 5.3: Choosing an Imputation Strategy

There are different ways to fill in missing values:

- **Mean Imputation**: Uses the average value (best for normally distributed data).
- **Median Imputation**: Uses the middle value (more robust to outliers and skewed data).
- **Mode Imputation**: Uses the most frequent value (best for categorical data).

```python

# Below are examples. Select one appropriate strategy per feature, not all at once.

# Impute 'bedrooms' using median
asheville_df['bedrooms'] = asheville_df['bedrooms'].fillna(asheville_df['bedrooms'].median())

# Impute 'bathrooms' using mean
asheville_df['bathrooms'] = asheville_df['bathrooms'].fillna(asheville_df['bathrooms'].mean())

# Impute 'host_is_superhost' using mode
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].fillna(
    asheville_df['host_is_superhost'].mode()[0]
) # # .mode()[0] is used because .mode() returns a Series. [0] extracts a single value (whether there’s one mode or many).
```





#### 🔍 What to Look For:

- **Which columns had missing values originally?**  

- **Did we drop rows only where `price` was missing?**  

- **Did missing values remain in other columns after dropping rows?**  

- **Were appropriate imputation strategies used per column?**  

- **Are all missing values now handled?**  

> Tip: Now that missing values are handled, inspect your dataset using .dtypes to verify column types and .describe() to explore statistical properties of numeric columns. This helps catch formatting or conversion issues before the next step

```python
# check data types
asheville_df.dtypes

# generate descriptive statistics
asheville_df.describe()

```


In [ ]:
# 5.1: Check for missing values in each column


In [ ]:
# 5.2: Drop rows where the target variable (price) is missing and then re-check for missing values


In [ ]:
# 5.3: Use one of the imputation methods to impute missing values, if there are any missing values after dropping records in step 2


In [ ]:
# check data types


In [ ]:
# generate descriptive statistics


### Exercise 6: Fixing Data Types and Encoding Columns

Now that we’ve handled missing values, let’s make sure our dataset has the correct data types and is ready for modeling. We’ll fix any incorrectly stored data types and encode categorical or boolean fields.

---

### 6.1: Identify Incorrect Data Types

We already checked `.dtypes` and `.describe()` in the previous exercise. Let’s now review what needs fixing:

- The `price` column is stored as `object` instead of numeric.
- Boolean-like columns (`host_identity_verified`, `host_is_superhost`) use `'t'` and `'f'` strings.
- The `room_type` column is categorical and may need encoding for machine learning.

---

### 6.2: Convert `price` to Numeric

```python
# Remove $ and convert to float
asheville_df['price'] = asheville_df['price'].replace(r'[\$,]', '', regex=True).astype(float)
```

---

### 6.3: Convert Boolean-like Columns

```python
# Convert 't'/'f' to 1/0
asheville_df['host_identity_verified'] = asheville_df['host_identity_verified'].map({'t': 1, 'f': 0})
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].map({'t': 1, 'f': 0})
```
---

### 6.4: Encode Categorical Column

```python
# One-hot encode 'room_type'
room_type_dummies = pd.get_dummies(asheville_df['room_type'], prefix='room_type', drop_first=True)
room_type_dummies = room_type_dummies.astype(int)

# Add new columns to the DataFrame
asheville_df = pd.concat([asheville_df, room_type_dummies], axis=1)

# Optional: Drop the original categorical column if you want
# asheville_df.drop('room_type', axis=1, inplace=True)
```



### 🔍 What to Look For:

- **Was the `price` column successfully converted to a numeric type?**  

- **Do the boolean columns now contain 1s and 0s instead of strings?**  

- **Did one-hot encoding work as expected?**  


> Use `.dtypes` to confirm column types and `.head()` to visually inspect the first few rows.


In [ ]:
# 6.2: Convert price to Numeric


In [ ]:
# 6.3: Convert Boolean-like Columns


In [ ]:
# 6.4: Encode Categorical Column



# Add new columns to the DataFrame


##4. Exploratory Data Analysis



###  Exercise 7: Exploratory Data Analysis (EDA)
Now that our dataset is cleaned and properly formatted, we can explore its structure and relationships using visual and statistical techniques.

---

## 7.1: Descriptive Statistics

Use `.describe()` to get a statistical summary of numeric features.

```python
asheville_df.describe()
```
---
## 7.2 : Histrograms of Distributions
Visualize distributions of numerical variables like price, bedrooms, bathrooms, and number_of_reviews.

```python
numeric_cols = ['price', 'bedrooms', 'bathrooms', 'number_of_reviews']
asheville_df[numeric_cols].hist(bins=30, figsize=(10,8))
plt.tight_layout()
plt.show()
```
## 7.3: Pair Plot
Understand relationships and potential clusters in the data..

```python
sns.pairplot(asheville_df[numeric_cols])
plt.show()
```

## 7.4: Correlation Matrix
See how variables relate to each other. Useful for feature selection and multicollinearity detection.

```python
corr = asheville_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()
```




### 7.5: Handling Outliers

e.g., As you may have observed from the earlier descriptive statistics and distribution plots, the `price` variable is highly skewed.

Most listings fall within a reasonable range, but a small number have unusually high prices. These outliers can distort visualizations and affect the performance of models later on.

To address this, we’ll use the **99th percentile rule** to filter out the top 1% of extreme values — a common practice when working with skewed real-world data.

```python
# Visualize the original price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (Before Removing Top 1%)")
plt.xlabel("Price")
plt.show()
```

```python
# Remove top 1% of extreme price values
price_cap = asheville_df['price'].quantile(0.99)
asheville_df = asheville_df[asheville_df['price'] <= price_cap]
```
```python
# Visualize the cleaned price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (After Removing Top 1%)")
plt.xlabel("Price")
plt.show()
```


### 7.6: Geospatial Plot

Visualize the geographic distribution of listings using latitude and longitude. Color can be used to represent listing price or categories.

We create **price buckets** to group listings into four categories:

- Less than $100  
- $100–$200  
- $200–$300  
- Greater than $300  

Using `plotly.express.scatter_mapbox`, we plot an **interactive map** that lets us:

- Zoom and pan around the city  
- Hover over listings to view price, bedroom, and bathroom details  
- Identify potential geographic pricing patterns (e.g., higher prices in city center vs outskirts)

```python
import plotly.express as px

# Create price buckets
asheville_df['price_range'] = pd.cut(
    asheville_df['price'],
    bins=[0, 100, 200, 300, asheville_df['price'].max()],
    labels=['< $100', '$100–$200', '$200–$300', '>$300']
)

# Plot the interactive map
fig = px.scatter_mapbox(
    asheville_df,
    lat='latitude',
    lon='longitude',
    color='price_range',
    hover_data=['price', 'bedrooms', 'bathrooms'],
    zoom=10,
    mapbox_style="open-street-map",
    title="Airbnb Listings in Asheville by Price Range"
)

fig.show()
```


🔍 What to Look For:

Use this section to uncover patterns, spot outliers, and understand relationships between variables. Key things to explore:

- **Are the numeric variables (like price, bedrooms, bathrooms) distributed normally or skewed?**

- **Are there any correlations between variables?**

- **Are there any surprising or extreme values (outliers)?**

- **Are there geographic patterns in pricing?**
  - Use the geospatial plot to spot high-price clusters or neighborhoods with dense listings.

Tip: EDA insights guide feature selection and transformation decisions before modeling. Take notes on variables that might need normalization, binning, winzorizing, or removal.


In [ ]:
# 7.1: Descriptive Statistics


In [ ]:
# 7.2: Histrograms of Distributions


In [ ]:
# 7.3: Pair Plot


In [ ]:
# correlation diagram


 7.5: Hadling price outliers

In [ ]:
# Visualize the original price distribution


In [ ]:
# Remove top 1% of extreme price values


In [ ]:
# Visualize the cleaned price distribution


In [ ]:
# 7.6: Geospatial Plot

